In [1]:
import os
import torch

from torch.amp import GradScaler, autocast
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.tensorboard import SummaryWriter
from tqdm import tqdm
from yacs.config import CfgNode as CN

from data.dataset import make_dataset
from src.utils import clean_exp_savedir
from src.losses import supervised_loss
import argparse

In [2]:
source_train_loader, target_train_loader, source_test_loader, target_test_loader = (
    make_dataset(
        source_dataset="office31_amazon",
        target_dataset="office31_dslr",
        img_size=384,
        train_bs=32,
        eval_bs=256,
        num_workers=16,
    )
)

In [3]:
import torch
import torch.nn as nn
import copy

from src.components.torch_nn import make_backbone, make_classifier_head
from src.components.visual_prompt import MultiHeadVisualPrompt

from src.utils import freeze_layers

class SingleModel(nn.Module):
    def __init__(
        self,
        backbone_type:str="vit_b_16",
        in_dim:int=768,
        hidden_dim:int=256,
        out_dim:int=31,
        imgsize:int=384,
        attribute_layers=[5,6,5,6],
        patch_size=[4,8,16,32],
        attribute_channels=3,
        dropout=[0.1,0.1,0.2,0.2],
        attr_net=["conv", "conv", "transformer", "transformer"],
        freeze_backbone=True
    ):
        super(SingleModel, self).__init__()
        self.backbone = make_backbone(backbone_type)
        self.backbone.fc = nn.Identity()
        if freeze_backbone:
            freeze_layers([self.backbone])
        
        self.visual_prompt = MultiHeadVisualPrompt(
            imgsize=imgsize, 
            layers=attribute_layers, 
            patch_size=patch_size, 
            channels=attribute_channels, 
            dropout=dropout, 
            attr_net=attr_net
        )
        self.classifier_head = make_classifier_head(
            in_dim=in_dim, 
            hidden_dim=hidden_dim, 
            out_dim=out_dim, 
            dropout=0.1, 
            type="class",
        )

    def forward(self, x: torch.Tensor, head_idx: list[int] | None=None):
        prompted_imgs = self.visual_prompt(x, head_idx)
        output_dict = {}
        for head, imgs in prompted_imgs.items():
            feat = self.backbone(imgs)
            logit = self.classifier_head(feat)
            output_dict[head] = {}
            output_dict[head]['feat'] = feat
            output_dict[head]['logit'] = logit
        return output_dict

class ModelEMA:
    """ Model Exponential Moving Average """
    def __init__(self, model, decay=0.999):
        self.ema = copy.deepcopy(model)
        self.ema.eval()
        self.decay = decay
        # Disable gradient tracking for the EMA model
        for param in self.ema.parameters():
            param.requires_grad_(False)

    def update(self, model):
        # Update EMA parameters
        with torch.no_grad():
            for ema_v, model_v in zip(self.ema.state_dict().values(), model.state_dict().values()):
                if ema_v.dtype.is_floating_point:
                    ema_v.copy_(ema_v * self.decay + (1. - self.decay) * model_v)

model = SingleModel(
    backbone_type="vit_b_32", 
    attribute_layers=[5,6,5,6],
    patch_size=[8,32,16,24]
)
device = torch.device("cuda")
model = model.to(device)
ema_model = ModelEMA(model, decay=0.9996)

Downloading: "https://github.com/lukemelas/PyTorch-Pretrained-ViT/releases/download/0.0.2/B_32_imagenet1k.pth" to /root/.cache/torch/hub/checkpoints/B_32_imagenet1k.pth
100%|██████████| 337M/337M [00:13<00:00, 27.1MB/s] 


Loaded pretrained weights.


In [4]:
scaler = GradScaler('cuda')
optimizer = torch.optim.AdamW(
    [
        {
            "params": list(model.classifier_head.parameters()),
            "lr": 1e-3,
            "weight_decay": 1e-4,
        },
        {
            "params": list(model.visual_prompt.parameters()),
            "lr": 5e-4,
            "weight_decay": 1e-5,
        },
    ]
)

epochs = 30
total_steps = epochs * len(source_train_loader)
scheduler = CosineAnnealingLR(optimizer, T_max=total_steps, eta_min=1e-5)

In [5]:
import torch
import torch.nn as nn

@torch.no_grad() 
def evaluate(model, branch, test_loader, device, criterion=nn.CrossEntropyLoss()):
    model.eval() 
    head_correct = {}
    head_loss = {}
    total_samples = 0
    
    for batch_data in test_loader:
        img, labels = batch_data
        img = img.to(device)
        labels = labels.to(device)
        batch_size = labels.size(0)
        total_samples += batch_size
        
        with torch.amp.autocast('cuda'): 
            output_dict = model(img)
            
        for head_id, sub_dict in output_dict.items():
            logits = sub_dict['logit']
            loss = criterion(logits, labels)
            
            # Calculate predictions
            _, preds = torch.max(logits, 1)
            correct = (preds == labels).sum().item()
            
            # Initialize dictionaries for new heads dynamically
            if head_id not in head_correct:
                head_correct[head_id] = 0
                head_loss[head_id] = 0.0
                
            head_correct[head_id] += correct
            # Multiply loss by batch size to get the true running sum
            head_loss[head_id] += loss.item() * batch_size 
            
    # Calculate and print final per-head metrics
    avg_total_loss = 0.0
    avg_total_acc = 0.0
    num_heads = len(head_correct)
    
    print(f"\n--- Detailed Evaluation ({branch} branch) ---")
    for head_id in head_correct.keys():
        h_acc = (head_correct[head_id] / total_samples) * 100
        h_loss = head_loss[head_id] / total_samples
        
        print(f"  Head {head_id} | Loss: {h_loss:.4f} | Accuracy: {h_acc:.2f}%")
        
        avg_total_loss += h_loss
        avg_total_acc += h_acc
        
    avg_total_loss /= num_heads
    avg_total_acc /= num_heads
    
    # Return averages to satisfy your training loop's expectation of two return values
    return avg_total_loss, avg_total_acc

In [6]:
os.makedirs("exp", exist_ok=True)
exp_save_dir = os.path.join("exp", "exp_1")
os.makedirs(exp_save_dir, exist_ok=True)
best_test_acc = 0
# Training loop
for epoch in range(epochs):
    running_loss = 0.0
    model.train()
    pbar = tqdm(
        source_train_loader,
        total=len(source_train_loader),
        desc=f"Epoch {epoch + 1}",
        ncols=100,
    )

    for batch_idx, source_data in enumerate(pbar):
        pbar.set_description_str(f"Epoch {epoch + 1}", refresh=True)
        current_step = epoch * len(source_train_loader) + batch_idx
        # weak_img, strong_img, label
        _, strong_img, src_labels = source_data 

        strong_img = strong_img.to(device)
        src_labels = src_labels.to(device)
        optimizer.zero_grad()
        loss = 0.0
        with autocast('cuda'):
            output_dict = model(strong_img)
            for head_id, sub_dict in output_dict.items():
                loss += supervised_loss(sub_dict['logit'], src_labels)
            running_loss += loss.item()
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        ema_model.update(model)

    test_loss_src, test_accuracy_src = evaluate(
        ema_model.ema, branch="src", test_loader=source_test_loader, device=device
    )
    test_loss_tgt, test_accuracy_tgt = evaluate(
         ema_model.ema, branch="tgt", test_loader=target_test_loader, device=device
    )
    
    print(
        f"Epoch [{epoch + 1}/{epochs}] Test Loss Source: {test_loss_src:.4f}, Test Accuracy Source: {test_accuracy_src:.2f}%"
    )
    print(
        f"Epoch [{epoch + 1}/{epochs}] Test Loss Target: {test_loss_tgt:.4f}, Test Accuracy Target: {test_accuracy_tgt:.2f}%"
    )

    if test_accuracy_src > best_test_acc:
        best_test_acc = test_accuracy_src
        ckpt_path = os.path.join(
            exp_save_dir, f"bi_best_{test_accuracy_src:.2f}.pth"
        )
        torch.save(
            {
                "epoch": epoch,
                "best_test_acc": best_test_acc,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scaler_state_dict": scaler.state_dict(),
            },
            ckpt_path,
        )
        print(f"New best checkpoint saved: {ckpt_path}")
        if test_accuracy_src == 100:
            break

Epoch 1: 100%|██████████████████████████████████████████████████████| 88/88 [00:41<00:00,  2.13it/s]



--- Detailed Evaluation (src branch) ---
  Head conv_0 | Loss: 3.3751 | Accuracy: 5.89%
  Head conv_1 | Loss: 3.3749 | Accuracy: 5.89%
  Head transformer_2 | Loss: 3.3749 | Accuracy: 5.89%
  Head transformer_3 | Loss: 3.3749 | Accuracy: 5.89%

--- Detailed Evaluation (tgt branch) ---
  Head conv_0 | Loss: 3.3650 | Accuracy: 12.05%
  Head conv_1 | Loss: 3.3650 | Accuracy: 12.05%
  Head transformer_2 | Loss: 3.3650 | Accuracy: 12.05%
  Head transformer_3 | Loss: 3.3650 | Accuracy: 12.05%
Epoch [1/30] Test Loss Source: 3.3750, Test Accuracy Source: 5.89%
Epoch [1/30] Test Loss Target: 3.3650, Test Accuracy Target: 12.05%
New best checkpoint saved: exp/exp_1/bi_best_5.89.pth


Epoch 2: 100%|██████████████████████████████████████████████████████| 88/88 [00:38<00:00,  2.28it/s]



--- Detailed Evaluation (src branch) ---
  Head conv_0 | Loss: 3.2464 | Accuracy: 16.86%
  Head conv_1 | Loss: 3.2464 | Accuracy: 16.90%
  Head transformer_2 | Loss: 3.2464 | Accuracy: 16.86%
  Head transformer_3 | Loss: 3.2464 | Accuracy: 16.93%

--- Detailed Evaluation (tgt branch) ---
  Head conv_0 | Loss: 3.2586 | Accuracy: 18.88%
  Head conv_1 | Loss: 3.2586 | Accuracy: 18.88%
  Head transformer_2 | Loss: 3.2596 | Accuracy: 18.88%
  Head transformer_3 | Loss: 3.2596 | Accuracy: 18.88%
Epoch [2/30] Test Loss Source: 3.2464, Test Accuracy Source: 16.89%
Epoch [2/30] Test Loss Target: 3.2591, Test Accuracy Target: 18.88%
New best checkpoint saved: exp/exp_1/bi_best_16.89.pth


Epoch 3: 100%|██████████████████████████████████████████████████████| 88/88 [00:38<00:00,  2.28it/s]


...


--- Detailed Evaluation (src branch) ---
  Head conv_0 | Loss: 0.2885 | Accuracy: 98.47%
  Head conv_1 | Loss: 0.2884 | Accuracy: 98.54%
  Head transformer_2 | Loss: 0.2625 | Accuracy: 98.62%
  Head transformer_3 | Loss: 0.2623 | Accuracy: 98.69%

--- Detailed Evaluation (tgt branch) ---
  Head conv_0 | Loss: 0.7279 | Accuracy: 86.35%
  Head conv_1 | Loss: 0.7258 | Accuracy: 86.95%
  Head transformer_2 | Loss: 0.7006 | Accuracy: 88.96%
  Head transformer_3 | Loss: 0.7049 | Accuracy: 88.55%
Epoch [28/30] Test Loss Source: 0.2755, Test Accuracy Source: 98.58%
Epoch [28/30] Test Loss Target: 0.7148, Test Accuracy Target: 87.70%
New best checkpoint saved: exp/exp_1/bi_best_98.58.pth


Epoch 29: 100%|█████████████████████████████████████████████████████| 88/88 [00:39<00:00,  2.24it/s]



--- Detailed Evaluation (src branch) ---
  Head conv_0 | Loss: 0.2601 | Accuracy: 98.54%
  Head conv_1 | Loss: 0.2601 | Accuracy: 98.54%
  Head transformer_2 | Loss: 0.2355 | Accuracy: 98.65%
  Head transformer_3 | Loss: 0.2354 | Accuracy: 98.69%

--- Detailed Evaluation (tgt branch) ---
  Head conv_0 | Loss: 0.6966 | Accuracy: 86.95%
  Head conv_1 | Loss: 0.6940 | Accuracy: 86.75%
  Head transformer_2 | Loss: 0.6703 | Accuracy: 88.76%
  Head transformer_3 | Loss: 0.6743 | Accuracy: 88.55%
Epoch [29/30] Test Loss Source: 0.2478, Test Accuracy Source: 98.61%
Epoch [29/30] Test Loss Target: 0.6838, Test Accuracy Target: 87.75%
New best checkpoint saved: exp/exp_1/bi_best_98.61.pth


Epoch 30: 100%|█████████████████████████████████████████████████████| 88/88 [00:38<00:00,  2.29it/s]



--- Detailed Evaluation (src branch) ---
  Head conv_0 | Loss: 0.2351 | Accuracy: 98.58%
  Head conv_1 | Loss: 0.2352 | Accuracy: 98.58%
  Head transformer_2 | Loss: 0.2119 | Accuracy: 98.69%
  Head transformer_3 | Loss: 0.2119 | Accuracy: 98.72%

--- Detailed Evaluation (tgt branch) ---
  Head conv_0 | Loss: 0.6685 | Accuracy: 86.95%
  Head conv_1 | Loss: 0.6659 | Accuracy: 86.95%
  Head transformer_2 | Loss: 0.6432 | Accuracy: 88.35%
  Head transformer_3 | Loss: 0.6467 | Accuracy: 88.55%
Epoch [30/30] Test Loss Source: 0.2235, Test Accuracy Source: 98.64%
Epoch [30/30] Test Loss Target: 0.6561, Test Accuracy Target: 87.70%
New best checkpoint saved: exp/exp_1/bi_best_98.64.pth


In [23]:
def evaluate_class_wise(model, head_id,head_name, test_loader, device, num_classes):
    model.eval()
    correct = 0
    total = 0
    total_loss = 0.0
    criterion = torch.nn.CrossEntropyLoss()

    correct_per_class = torch.zeros(num_classes, device=device)
    total_per_class = torch.zeros(num_classes, device=device)

    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)

            pred = model(images, head_id)[head_name]['logit']
            loss = criterion(pred, labels)
            
            total_loss += loss.item() * images.size(0)
            _, predicted = torch.max(pred, 1)
            
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

            # Update class-wise metrics for the current batch
            for c in range(num_classes):
                class_mask = (labels == c)
                total_per_class[c] += class_mask.sum()
                correct_per_class[c] += (predicted[class_mask] == labels[class_mask]).sum()

    avg_loss = total_loss / total
    accuracy = 100 * correct / total
    
    # Calculate class-wise accuracy (in percentage) and handle division by zero
    class_wise_accuracy = torch.where(
        total_per_class > 0, 
        (correct_per_class / total_per_class) * 100, 
        torch.tensor(0.0, device=device)
    )

    return avg_loss, accuracy, class_wise_accuracy

In [7]:
class DSSD_SignalExtractor(nn.Module):
    def __init__(self, num_radial_bins=32, num_angle_bins=18):
        super().__init__()
        self.num_radial_bins = num_radial_bins
        self.num_angle_bins = num_angle_bins
        
        sobel_x = torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]]).view(1, 1, 3, 3)
        sobel_y = torch.tensor([[-1., -2., -1.], [0., 0., 0.], [1., 2., 1.]]).view(1, 1, 3, 3)
        self.register_buffer('sobel_x', sobel_x)
        self.register_buffer('sobel_y', sobel_y)
    def rgb_to_gray(self, x):
        if x.shape[1]==3: 
            x = 0.2989 * x[:, 0:1] + 0.5870 * x[:, 1:2] + 0.1140 * x[:, 2:3]
        return x

    def extract_texture_fourier(self, x):
        """
        Extracts texture using 1D Radial Power Spectrum of the Fourier Transform.
        """
        x = self.rgb_to_gray(x) 
        B, C, H, W = x.shape
        fft2d = torch.fft.fft2(x)
        fftshift = torch.fft.fftshift(fft2d)
        power_spectrum = torch.abs(fftshift) ** 2
        
        y, x_coord = torch.meshgrid(torch.arange(H), torch.arange(W), indexing='ij')
        center_y, center_x = H // 2, W // 2
        radius = torch.sqrt((y - center_y)**2 + (x_coord - center_x)**2).to(x.device)
        
        max_radius = min(center_y, center_x)
        radial_profile = torch.zeros((B, self.num_radial_bins), device=x.device)
        
        bin_edges = torch.linspace(0, max_radius, self.num_radial_bins + 1, device=x.device)
        for i in range(self.num_radial_bins):
            mask = (radius >= bin_edges[i]) & (radius < bin_edges[i+1])
            radial_profile[:, i] = power_spectrum[:, 0, mask].mean(dim=1)
            
        radial_profile = F.normalize(radial_profile, p=2, dim=1)
        return radial_profile

    def extract_shape_gradients(self, x):
        """
        Extracts shape using a lightweight Histogram of Oriented Gradients (HOG) approximation.
        """
        x = self.rgb_to_gray(x)
        grad_x = F.conv2d(x, self.sobel_x, padding=1)
        grad_y = F.conv2d(x, self.sobel_y, padding=1)
        
        magnitude = torch.sqrt(grad_x**2 + grad_y**2 + 1e-6)
        angle = torch.atan2(grad_y, grad_x) # Range: [-pi, pi]
        
        B = x.shape[0]
        shape_profile = torch.zeros((B, self.num_angle_bins), device=x.device)
        angle_bins = torch.linspace(-torch.pi, torch.pi, self.num_angle_bins + 1, device=x.device)
        
        for i in range(self.num_angle_bins):
            mask = (angle >= angle_bins[i]) & (angle < angle_bins[i+1])
            shape_profile[:, i] = (magnitude * mask).view(B, -1).sum(dim=1)
            
        shape_profile = F.normalize(shape_profile, p=2, dim=1)
        return shape_profile

In [21]:
import torch.nn.functional as F
class SourceStatistics: 
    def __init__(self, tex_mu,  tex_inv_cov, shp_mu, shp_inv_cov):
        self.tex_mu = tex_mu
        self.tex_inv_cov = tex_inv_cov
        self.shp_mu = shp_mu
        self.shp_inv_cov = shp_inv_cov

    def to(self, device: torch.device) -> "SourceStatistics":
        return SourceStatistics(
            self.tex_mu.to(device),
            self.tex_inv_cov.to(device),
            self.shp_mu.to(device),
            self.shp_inv_cov.to(device),
        )
 
    @property
    def num_classes(self) -> int:
        return self.tex_mu.shape[0]
  
    def mahalanobis_tex(self, x: torch.Tensor) -> torch.Tensor:
        """x: [B, d_tex]  →  [B, C]  distance from each sample to each class."""
        return self._batch_mahalanobis(x, self.tex_mu, self.tex_inv_cov)
 
    def mahalanobis_shp(self, x: torch.Tensor) -> torch.Tensor:
        """x: [B, d_shp]  →  [B, C]  distance from each sample to each class."""
        return self._batch_mahalanobis(x, self.shp_mu, self.shp_inv_cov)

    @staticmethod
    def _batch_mahalanobis(
        x:       torch.Tensor,   # [B, d]
        mu:      torch.Tensor,   # [C, d]
        inv_cov: torch.Tensor,   # [C, d, d]
    ) -> torch.Tensor:           # [B, C]
        delta = x.unsqueeze(1) - mu.unsqueeze(0)          # [B, C, d]
        left  = torch.einsum('bcd,cde->bce', delta, inv_cov)  # [B, C, d]
        dist  = (left * delta).sum(dim=-1).clamp(min=1e-8).sqrt()  # [B, C]
        return dist

In [22]:
extractor = DSSD_SignalExtractor(64, 16).to("cuda")

def compute_source_statistics(source_loader, extractor, num_classes, device, epsilon = 1e-5) -> SourceStatistics:
    extractor.eval()
    all_tex, all_shp, all_labels = [], [], []
    with torch.no_grad():
        for images, labels in source_loader:
            images = images.to(device)
            all_tex.append(extractor.extract_texture_fourier(images).cpu())
            all_shp.append(extractor.extract_shape_gradients(images).cpu())
            all_labels.append(labels)
 
    all_tex    = torch.cat(all_tex,    dim=0)   # [N_src, d_tex]
    all_shp    = torch.cat(all_shp,    dim=0)   # [N_src, d_shp]
    all_labels = torch.cat(all_labels, dim=0)   # [N_src]
 
    C, d_tex, d_shp = num_classes, all_tex.shape[1], all_shp.shape[1]
 
    tex_mu      = torch.zeros(C, d_tex,         device='cpu')
    tex_inv_cov = torch.zeros(C, d_tex, d_tex,  device='cpu')
    shp_mu      = torch.zeros(C, d_shp,         device='cpu')
    shp_inv_cov = torch.zeros(C, d_shp, d_shp,  device='cpu')
 
    for c in range(C):
        mask = (all_labels == c)
        f_t = all_tex[mask]                                # [n_c, d_tex]
        if f_t.shape[0] >= 2:
            mu_t  = f_t.mean(0)
            ctr_t = f_t - mu_t
            cov_t = (ctr_t.T @ ctr_t) / (f_t.shape[0] - 1)
            cov_t += torch.eye(d_tex) * epsilon
            tex_mu[c]      = mu_t
            tex_inv_cov[c] = torch.linalg.inv(cov_t)
 
        f_s = all_shp[mask]                                # [n_c, d_shp]
        if f_s.shape[0] >= 2:
            mu_s  = f_s.mean(0)
            ctr_s = f_s - mu_s
            cov_s = (ctr_s.T @ ctr_s) / (f_s.shape[0] - 1)
            cov_s += torch.eye(d_shp) * epsilon
            shp_mu[c]      = mu_s
            shp_inv_cov[c] = torch.linalg.inv(cov_s)
  
    stats = SourceStatistics(tex_mu, tex_inv_cov, shp_mu, shp_inv_cov)
    return stats.to(device)

stats =compute_source_statistics(source_loader= source_test_loader, extractor=extractor, num_classes=31, device=device)

In [11]:
class RoutingFunction(nn.Module):
    def __init__(
        self,
        num_heads:  int,
        head_types,
        d_tex:      int,         # texture feature dim (e.g. 32)
        d_shp:      int,         # shape feature dim   (e.g. 18)
        K:          int,         # number of sparse heads to activate
        hidden_dim: int = 64,
    ):
        super().__init__()
        assert K <= num_heads, f"K={K} cannot exceed num_heads={num_heads}"
 
        self.num_heads = num_heads
        self.head_types = head_types
        self.K = K
 
        # Fixed masks broadcast prior score to each head by its modality type
        tex_mask = torch.tensor([1.0 if t == 'texture' else 0.0 for t in head_types])
        shp_mask = torch.tensor([1.0 if t == 'shape'   else 0.0 for t in head_types])
        self.register_buffer('tex_mask', tex_mask)   # [N]
        self.register_buffer('shp_mask', shp_mask)   # [N]
 
        # Learnable MLP: (tex_feat || shp_feat || prior) → routing correction
        mlp_in = d_tex + d_shp + num_heads
        self.mlp = nn.Sequential(
            nn.Linear(mlp_in, hidden_dim),
            nn.GELU(),
            nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, num_heads),
        )
        # Init MLP output near zero so it starts close to the pure stats prior
        nn.init.zeros_(self.mlp[-1].weight)
        nn.init.zeros_(self.mlp[-1].bias)
  
    def _stats_prior(
        self,
        tex_feat: torch.Tensor,   # [B, d_tex]
        shp_feat: torch.Tensor,   # [B, d_shp]
        D: SourceStatistics,
    ) -> torch.Tensor:             # [B, N] unnormalized scores
        """
        Score each head based on how well the sample fits the source statistics.
        Negative distance = higher score (closer to source distribution = preferred).
 
        We marginalize over classes using min distance (best-case class):
          tex_score_b = -min_c Mahalanobis_tex(tex_feat_b, mu_tex[c], inv_cov_tex[c])
        Then broadcast to each head via its modality type mask.
        """
        tex_dist = D.mahalanobis_tex(tex_feat)   # [B, C]
        shp_dist = D.mahalanobis_shp(shp_feat)   # [B, C]
 
        # Best-case class distance per modality: [B, 1]
        tex_score = -tex_dist.min(dim=1, keepdim=True).values   # [B, 1]
        shp_score = -shp_dist.min(dim=1, keepdim=True).values   # [B, 1]
 
        # Broadcast to each head by type: [B, N]
        prior = tex_score * self.tex_mask + shp_score * self.shp_mask
        return prior
  
    def forward(
        self,
        tex_feat: torch.Tensor,   # [B, d_tex]
        shp_feat: torch.Tensor,   # [B, d_shp]
        D: SourceStatistics,
    ):
        """
        Returns
        -------
        r_soft  : [B, N]  soft routing weights (differentiable, rows sum to 1)
        gate_id : [B, K]  top-K head indices   (non-differentiable)
        """
        prior  = self._stats_prior(tex_feat, shp_feat, D)          # [B, N]
        mlp_in = torch.cat([tex_feat, shp_feat, prior], dim=-1)     # [B, d_tex+d_shp+N]
        logits = self.mlp(mlp_in) + prior                           # residual on prior
 
        r_soft  = F.softmax(logits, dim=-1)                         # [B, N]
        _, gate_id = logits.topk(self.K, dim=-1)                    # [B, K]
 
        return r_soft, gate_id
 
    def forward_soft_only(
        self,
        tex_feat: torch.Tensor,
        shp_feat: torch.Tensor,
        D: SourceStatistics,
    ) -> torch.Tensor:
        """Returns only r_soft — used for the strong-aug differentiable pass."""
        r_soft, _ = self.forward(tex_feat, shp_feat, D)
        return r_soft
 

In [12]:
x_tgt_w, x_tgt_s, _ = next(iter(target_train_loader))
x_tgt_w = x_tgt_w.to(device)
tex_feat = extractor.extract_texture_fourier(x_tgt_w)
shp_feat = extractor.extract_shape_gradients(x_tgt_w)

In [26]:
router = RoutingFunction(num_heads=4, head_types=['texture', 'texture', 'shape', 'shape'], d_tex= 64,  d_shp=16,  K=1, hidden_dim = 128).to(device)
r_soft_w, gate_id = router(tex_feat, shp_feat, stats)

In [27]:
r_soft_w

tensor([[4.6212e-01, 4.6212e-01, 3.7881e-02, 3.7881e-02],
        [4.9846e-01, 4.9846e-01, 1.5427e-03, 1.5427e-03],
        [4.4104e-01, 4.4104e-01, 5.8964e-02, 5.8964e-02],
        [4.0658e-01, 4.0658e-01, 9.3420e-02, 9.3420e-02],
        [4.9987e-01, 4.9987e-01, 1.2646e-04, 1.2646e-04],
        [4.9322e-01, 4.9322e-01, 6.7826e-03, 6.7826e-03],
        [5.0625e-02, 5.0625e-02, 4.4938e-01, 4.4938e-01],
        [4.0147e-02, 4.0147e-02, 4.5985e-01, 4.5985e-01],
        [3.3236e-01, 3.3236e-01, 1.6764e-01, 1.6764e-01],
        [4.6721e-01, 4.6721e-01, 3.2794e-02, 3.2794e-02],
        [1.4730e-01, 1.4730e-01, 3.5270e-01, 3.5270e-01],
        [4.9769e-01, 4.9769e-01, 2.3058e-03, 2.3058e-03],
        [4.6711e-01, 4.6711e-01, 3.2890e-02, 3.2890e-02],
        [4.7909e-01, 4.7909e-01, 2.0909e-02, 2.0909e-02],
        [2.1214e-01, 2.1214e-01, 2.8786e-01, 2.8786e-01],
        [2.1373e-01, 2.1373e-01, 2.8627e-01, 2.8627e-01],
        [4.3673e-01, 4.3673e-01, 6.3265e-02, 6.3265e-02],
        [3.488

In [29]:
out = model(x_tgt_s, gate_id)

TypeError: only integer tensors of a single element can be converted to an index